# 04 Visualization

This notebook audits the graph drawing API: forward graphs, rolled/unrolled modes, node customization, module focus, backward graphs, combined graphs, and a code-panel example. The cells save small SVG files under `notebooks/audit/_artifacts/` so outputs stay reviewable.

We use a compact model with named modules. The output is scalar-friendly for backward graph capture.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import torch
from torch import nn
import torchlens as tl
from IPython.display import SVG, display
from torchlens.experimental.dagua import NodeSpec

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "audit" / "_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(19)


class VizNet(nn.Module):
    """Small model for TorchLens visualization examples."""

    def __init__(self) -> None:
        """Initialize named modules."""
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(4, 5), nn.ReLU())
        self.head = nn.Linear(5, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run one forward pass."""
        h = self.encoder(x)
        return self.head(h).sum()


model = VizNet().eval()
x = torch.randn(2, 4, requires_grad=True)
trace = tl.trace(model, x, gradients_to_save="all", save_code_context=True)
print(trace.summary(fields=["name", "shape", "params"]))

`Trace.draw()` renders the forward graph. `vis_save_only=True` writes the file and returns DOT source so we can verify the render without embedding a large image.

In [ ]:
forward_path = ARTIFACT_DIR / "04_forward_graph"
forward_dot = trace.draw(
    vis_outpath=str(forward_path),
    vis_save_only=True,
    vis_fileformat="svg",
)
print(f"DOT starts with: {forward_dot.splitlines()[0]}")
print(
    f"saved: {forward_path.with_suffix('.svg')} ({forward_path.with_suffix('.svg').stat().st_size} bytes)"
)
display(SVG(filename=str(forward_path.with_suffix(".svg"))))
print("note: this first render is the default forward graph.")

Rolled and unrolled modes are both available. This model has no recurrence, so both render cleanly and should be similar in size.

In [ ]:
for mode in ["rolled", "unrolled"]:
    outpath = ARTIFACT_DIR / f"04_{mode}_graph"
    dot = trace.draw(
        vis_mode=mode,
        vis_outpath=str(outpath),
        vis_save_only=True,
        vis_fileformat="svg",
    )
    print(f"{mode:8s}: {len(dot)} DOT chars, file={outpath.with_suffix('.svg').name}")
print("note: rolled and unrolled match here because this tiny model has no recurrence.")

Node specs let users customize graph labels and styling. This example highlights `linear` nodes while leaving other nodes at their defaults.

In [ ]:
def highlight_linear(layer: tl.Layer, default_spec: NodeSpec) -> NodeSpec | None:
    """Highlight linear layers and keep the default for other layers."""
    del default_spec
    if layer.layer_type == "linear":
        return NodeSpec(lines=[layer.layer_label, "Linear op"], fillcolor="#fff2a8")
    return None


node_spec_path = ARTIFACT_DIR / "04_node_spec_graph"
node_spec_dot = trace.draw(
    node_spec_fn=highlight_linear,
    vis_outpath=str(node_spec_path),
    vis_save_only=True,
    vis_fileformat="svg",
)
print("custom label present:", "Linear op" in node_spec_dot)
print(f"saved: {node_spec_path.with_suffix('.svg')}")
display(SVG(filename=str(node_spec_path.with_suffix(".svg"))))
print("note: linear nodes are highlighted by the custom NodeSpec.")

Module focus renders a specific submodule region. Here we focus on the `encoder` module path.

In [ ]:
module_path = ARTIFACT_DIR / "04_encoder_focus_graph"
module_dot = trace.draw(
    module="encoder",
    vis_outpath=str(module_path),
    vis_save_only=True,
    vis_fileformat="svg",
)
print("encoder mentioned in DOT:", "encoder" in module_dot)
print(f"saved: {module_path.with_suffix('.svg')}")
display(SVG(filename=str(module_path.with_suffix(".svg"))))
print("note: module focus limits the render to the encoder region.")

Backward visualization requires an explicit backward capture. We log backward from the saved scalar output, then render the autograd graph.

In [ ]:
loss = trace[trace.output_layers[0]].out
trace.log_backward(loss)
backward_path = ARTIFACT_DIR / "04_backward_graph"
backward_dot = trace.draw_backward(
    vis_outpath=str(backward_path),
    vis_save_only=True,
    vis_fileformat="svg",
)
print("backward graph rendered:", "backward graph" in backward_dot)
print(f"grad fns captured: {len(trace.grad_fns)}")
print(f"saved: {backward_path.with_suffix('.svg')}")
display(SVG(filename=str(backward_path.with_suffix(".svg"))))
print("note: backward render adds captured autograd nodes after log_backward().")

Combined graphs place forward ops and backward grad functions in one view. This is useful when auditing how a forward layer connects to its gradient path.

In [ ]:
combined_path = ARTIFACT_DIR / "04_combined_graph"
combined_dot = trace.draw_combined(
    vis_outpath=str(combined_path),
    vis_save_only=True,
    vis_fileformat="svg",
)
combined_svg = combined_path.with_suffix(".svg")
combined_ok = combined_svg.exists() and combined_svg.stat().st_size > 0 and len(trace.grad_fns) > 0
if not combined_ok:
    raise RuntimeError("combined graph did not render a non-empty SVG")
print("combined graph rendered:", combined_ok)
print(f"saved: {combined_svg} ({combined_svg.stat().st_size} bytes)")
display(SVG(filename=str(combined_svg)))
print("note: combined render shows forward layers together with backward GradFn records.")

The code panel can include source text in the graph. In notebook execution, built-in source capture can be environment-dependent, so this audit uses the documented callable form to supply source text directly.

In [ ]:
code_panel_path = ARTIFACT_DIR / "04_code_panel_graph"
code_panel_dot = trace.draw(
    code_panel=lambda model: "class VizNet(nn.Module):\n    def forward(self, x): ...",
    vis_outpath=str(code_panel_path),
    vis_save_only=True,
    vis_fileformat="svg",
)
print("code panel rendered:", "cluster_torchlens_code_panel" in code_panel_dot)
print(f"saved: {code_panel_path.with_suffix('.svg')}")
display(SVG(filename=str(code_panel_path.with_suffix(".svg"))))
print("note: code-panel render adds source text beside the graph.")

## Tensor Visualizers and Top-Level Graph Wrappers
`tl.viz.heatmap`, `tl.viz.channel_grid`, and `tl.viz.histogram` return PIL images for tensor payloads. The package-level `tl.draw_backward`, `tl.draw_combined`, and `tl.show_bundle_graph` wrappers are also available in this build as deprecated compatibility shims; expected deprecation warnings are suppressed so the visible output stays focused on generated artifacts.


In [ ]:
import warnings

tiny_activation = torch.arange(3 * 4 * 4, dtype=torch.float32).reshape(1, 3, 4, 4)
visualizers = {
    "heatmap": tl.viz.heatmap(max_size=120),
    "channel_grid": tl.viz.channel_grid(n=3, max_size=120),
    "histogram": tl.viz.histogram(bins=8, width=160, height=100),
}
for name, visualizer in visualizers.items():
    image = visualizer(tiny_activation)
    if image is None:
        print(f"> NOTE: tl.viz.{name} returned None for this tensor shape.")
    else:
        path = ARTIFACT_DIR / f"04_tensor_{name}.png"
        image.save(path)
        print(f"tl.viz.{name}: size={image.size} saved={path.name}")

if hasattr(tl, "draw_backward"):
    top_backward_path = ARTIFACT_DIR / "04_top_level_backward"
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        tl.draw_backward(
            trace, vis_outpath=str(top_backward_path), vis_save_only=True, vis_fileformat="svg"
        )
    print(
        f"tl.draw_backward compatibility shim saved: {top_backward_path.with_suffix('.svg').exists()}"
    )
else:
    print("> NOTE: tl.draw_backward is not exported in this build.")

if hasattr(tl, "draw_combined"):
    top_combined_path = ARTIFACT_DIR / "04_top_level_combined"
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        tl.draw_combined(
            trace, vis_outpath=str(top_combined_path), vis_save_only=True, vis_fileformat="svg"
        )
    print(
        f"tl.draw_combined compatibility shim saved: {top_combined_path.with_suffix('.svg').exists()}"
    )
else:
    print("> NOTE: tl.draw_combined is not exported in this build.")

if hasattr(tl, "show_bundle_graph"):
    bundle_graph_path = ARTIFACT_DIR / "04_bundle_graph"
    second_trace = tl.trace(model, x + 0.1)
    tiny_bundle = tl.bundle({"base": trace, "shifted": second_trace}, baseline="base")
    try:
        tl.show_bundle_graph(
            tiny_bundle,
            vis_outpath=str(bundle_graph_path),
            vis_save_only=True,
            vis_fileformat="svg",
        )
        print(f"tl.show_bundle_graph saved: {bundle_graph_path.with_suffix('.svg').exists()}")
    except Exception as exc:
        print(f"> NOTE: tl.show_bundle_graph gap: {type(exc).__name__}: {exc}")
else:
    print("> NOTE: tl.show_bundle_graph is not exported in this build.")

## 🔧 Sandbox

Mini-experiment: change `sandbox_mode`, `sandbox_module`, and `sandbox_node_style`. Expected observation: focusing on a module reduces the DOT/SVG size, and node style changes only presentation. The cell prints before/after size deltas.

In [ ]:
sandbox_mode = "unrolled"
sandbox_module = "encoder"
sandbox_node_style = "profiling"

baseline_svg = forward_path.with_suffix(".svg")
sandbox_path = ARTIFACT_DIR / "04_sandbox_graph"
sandbox_dot = trace.draw(
    vis_mode=sandbox_mode,
    module=sandbox_module,
    node_style=sandbox_node_style,
    vis_outpath=str(sandbox_path),
    vis_save_only=True,
    vis_fileformat="svg",
)
sandbox_svg = sandbox_path.with_suffix(".svg")
print(f"mode={sandbox_mode}, module={sandbox_module}, node_style={sandbox_node_style}")
print(f"DOT chars: {len(forward_dot)} -> {len(sandbox_dot)}")
print(f"SVG bytes: {baseline_svg.stat().st_size} -> {sandbox_svg.stat().st_size}")
print(f"byte delta: {sandbox_svg.stat().st_size - baseline_svg.stat().st_size}")
display(SVG(filename=str(sandbox_svg)))